# Lektion 18: Sikring af AI-agenter med kryptografiske kvitteringer

## Praktisk notesbog

Denne notesbog gennemgår fire øvelser:

1. **Signer din første kvittering** for et agentværktøjskald og verificer den.
2. **Manipuler med kvitteringen** og se verificeringen fejle.
3. **Byg en kæde af tre kvitteringer** og bekræft kædens integritet.
4. **Indpak et Microsoft Agent Framework-værktøjskald**, så hver handling udsteder en kvittering.

Alle kryptografiske primitive importeres fra velholdte biblioteker (`pynacl` for Ed25519, `jcs` for RFC 8785 kanonisk JSON, `hashlib` fra Pythons standardbibliotek for SHA-256). Selve kvitteringslogikken er almindelig Python, som du kan læse og modificere.

Kør cellerne i rækkefølge. Hver sektion er kort og selvstændig.


## Setup

Installer de to afhængigheder. Begge har tilladende licenser (Apache-2.0 / MIT).


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## Hjælpeværktøjer

Disse to hjælpere håndterer base64url-kodning (uden padding) og SHA-256-hashning af vilkårlige objekter. De holder resten af notebogen fokuseret på kvitteringslogikken selv.


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Section 1: Underskriv din første kvittering

Forestil dig, at vores agent for **Contoso Travel** lige har slået flyvninger op fra Sydney til Los Angeles for en kunde. Vi vil registrere dette værktøjskald som en underskrevet kvittering, så en fremtidig revisor kan verificere det uden at skulle stole på os.

### Step 1.1: Generer en underskrivningsnøgle

I produktion ville agentens underskrivningsnøgle være placeret i en hardware-sikkerhedsmodul (HSM), Azure Key Vault eller en lignende beskyttet lagring. Til denne lektion genererer vi en frisk nøgle i hukommelsen.


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### Trin 1.2: Byg kvitteringsindholdet

Indholdet indeholder alt det, vi ønsker, at kvitteringen skal bekræfte: hvem der handlede, hvilket værktøj, med hvilke argumenter, hvad der kom tilbage, under hvilken politik, og hvornår. Vi hasher argumenterne og resultatet i stedet for at inkludere dem direkte, så kvitteringen ikke lækker følsomt indhold.


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### Trin 1.3: Underskriv og saml kvitteringen

Tre trin:

1. Kanoniser payloaden ved hjælp af JCS, så to implementeringer, der producerer den samme logiske kvittering, genererer byte-identiske bytes.
2. Hash de kanoniske bytes med SHA-256.
3. Underskriv hashen med Ed25519 private nøgle.

Signaturen vedhæftes derefter den oprindelige payload for at producere den endelige kvittering.


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### Trin 1.4: Bekræft kvitteringen

Bekræftelsen omvender processen. Vi fjerner signaturen, genberegner den kanoniske hash, og tjekker signaturen mod den offentlige nøgle i kvitteringen.

En revisor, der udfører denne bekræftelse, har ikke brug for noget fra os undtagen selve kvitteringen. Ingen tjeneste at kalde, ingen nøglemappe at forespørge, ingen tillid krævet.


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

Du bør se `Receipt is valid: True`. Agenten har produceret sin første kryptografisk signerede revisionspost.


## Sektion 2: Manipulation med kvitteringen

Hele pointen med kvitteringer er, at de er manipulationssikre. Lad os bevise det.

Vi vil ændre præcis ét tegn på kvitteringen og se verifikationen fejle.


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### Hvad skete der lige?

Da vi ændrede `policy_id`, ændrede de kanoniske bytes sig. SHA-256-hashen af disse bytes ændrede sig. Signaturen (som var over den oprindelige hash) matcher ikke længere den nye hash. Verifikationen returnerer korrekt `False`.

Der er ingen måde at ændre nogen felt i kvitteringen på og stadig få den til at verificere, medmindre angriberen har den private nøgle. Så længe den private nøgle opbevares i en nøgleboks, og den offentlige nøgle er offentliggjort, er manipulation umulig at skjule.

Prøv det selv: ændr `tool_name` eller `agent_id` eller `timestamp` i cellen ovenfor og kør igen. Hver ændring giver en ugyldig kvittering.


## Sektion 3: Kæd kvitteringer sammen

En enkelt kvittering beskytter én handling. De fleste agenter udfører mange handlinger. For at gøre hele sekvensen manipulationssikker, kæder vi hver kvittering til den forrige ved at inkludere den forrige kvitterings hash i den nye kvitterings payload.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

Hvis nogen fjerner eller omrokerer en kvittering, brydes kæden præcis på det punkt. Verificeringen af enhver senere kvittering mislykkes, fordi dens `previous_receipt_hash` ikke længere matcher den faktiske hash af dens forgænger.


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Bryde nu kæden ved at manipulere med den midterste kvittering og verificere igen. Den manipulerede kvittering fejler sin signaturkontrol, OG den næste kvittering fejler sin kædelinkkontrol (fordi dens `previous_receipt_hash` ikke længere matcher den ændrede midterste kvitterings hash).


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Kvittering 0 verificerer stadig (den blev ikke ændret og har ingen forudgående kvittering, den afhænger af). Kvittering 1 fejler sin signaturkontrol, fordi vi ændrede `tool_args_hash`. Kvittering 2 fejler sin kædelink-kontrol, fordi dens `previous_receipt_hash` blev beregnet ud fra den oprindelige (nu ændrede) kvittering 1.

Selv hvis en angriber underskriver den ændrede kvittering 1 igen (hvilket de ikke kan gøre uden den private nøgle), ville kædelink-ubalance i kvittering 2 stadig afsløre manipuleringsforsøget. For at skjule ændringen skulle angriberen genunderskrive hver kvittering fra ændringspunktet og frem, hvilket kræver besiddelse af den private nøgle.


## Sektion 4: Indpak en agentværktøjskald med kvitteringssignering

I en rigtig implementering ønsker du ikke, at hver agentforfatter skal huske at kalde `make_receipt`. Du vil have, at kvitteringssignering skal være automatisk for hvert værktøjskald.

Her er det simpleste mønster: en wrapper-klasse, der tager en hvilken som helst kaldbar værktøjsfunktion og returnerer en kvitteringsudstedende version af den. Dette tilpasser sig til enhver agentramme, inklusive Microsoft Agent Framework (`agent_framework.azure`).

Hvis du ikke har et Azure AI Foundry-projekt sat op, demonstrerer den lokale mock herunder stadig mønsteret.


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to AzureAIProjectAgentProvider as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")

### Integration med Microsoft Agent Framework

`ReceiptedTool`-indpakningen ovenfor er frameworks-agnostisk. For at bruge den inden for en agent bygget med Microsoft Agent Framework, registrer den indpakkede funktion som et værktøj. En skitse (du ville erstatte mock-up'en med en rigtig Azure AI Foundry værktøjsregistrering):

```python
# Pseudokode, der viser integrationsformen.
# fra agent_framework.azure import AzureAIProjectAgentProvider
# fra azure.identity import AzureCliCredential
#
# provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())
# agent = provider.create_agent(
#     instructions="Du er en Contoso Travel-agent ...",
#     tools=[receipted_lookup],   # det indkapslede værktøj, ikke den rå funktion
# )
# response = agent.run("Find flyrejser fra Sydney til Los Angeles i juni.")
#
# # Efter kørslen har hvert værktøj, agenten brugte, en underskrevet kvittering:
# audit_chain = receipted_lookup.receipts
```

Agent-frameworket behøver ikke at vide noget om kvitteringer. Kvitteringssignering er indpakket omkring værktøjet, ikke skruet fast i frameworket. Dette er, hvordan du tilføjer oprindelse til eksisterende agentkode uden at omskrive agenten.


## Opsummering og stræk-udfordring

Du har:

- Genereret et Ed25519 nøglepar.
- Oprettet og underskrevet en kvittering for et agentværktøjskald.
- Verificeret kvitteringen offline ved kun at bruge den offentlige nøgle.
- Manipuleret med en kvittering og observeret verifikationsfejl.
- Oprettet en hash-kædet sekvens af tre kvitteringer.
- Manipuleret midten af kæden og observeret både signaturfejl og kædelink-fejl.
- Gennemført automatisk kvitteringssignering ved pakning af en værktøjsfunktion.

**Stræk-udfordring.** Udvid kvitteringsskemaet med et `request_id` felt (en UUID til distribueret sporing). Opdater `make_receipt` til at inkludere det, og bekræft at kvitteringer stadig verificeres korrekt fra ende til anden. Ændr derefter feltet efter signering og bekræft at verifikationen fejler. Dette tvinger dig til at internalisere, hvordan hver eneste byte af den kanoniske kodning bidrager til signaturen.

**Vigtigt afgrænsningspunkt.** Kvitteringer beviser tre ting, og kun tre ting: tilskrivning (denne nøgle har underskrevet dette indhold), integritet (indholdet er ikke ændret siden signering), og rækkefølge (denne kvittering kom efter den anden kvittering). De beviser IKKE, at agentens handling var korrekt, at politikken navngivet i `policy_id` faktisk blev evalueret, eller at agenten fulgte alle reglerne. Kvitteringer er et fundament. Styring er det system, du bygger ovenpå.

Læs lektionens README igen med det afgrænsningspunkt i tankerne. Den mest almindelige fejl hold laver med kvitteringer er at antage, at "vi har kvitteringer" betyder "vi er styret." Det gør det ikke. Kvitteringer gør agentadfærd reviderbar. De gør den ikke korrekt.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfraskrivelse**:
Dette dokument er blevet oversat ved hjælp af AI-oversættelsestjenesten [Co-op Translator](https://github.com/Azure/co-op-translator). Selvom vi bestræber os på nøjagtighed, skal du være opmærksom på, at automatiserede oversættelser kan indeholde fejl eller unøjagtigheder. Det originale dokument på dets oprindelige sprog bør betragtes som den autoritative kilde. For kritisk information anbefales professionel menneskelig oversættelse. Vi påtager os intet ansvar for misforståelser eller fejltolkninger, der opstår som følge af brugen af denne oversættelse.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
